# 00 — Demand-model and merger-simulation functions

Loaded by notebooks 03–08 with `%run 00_model_functions.ipynb` (after `00_setup.ipynb`).

* `add_instruments` builds candidate excluded instruments: BLP sums of rival characteristics, numbers of
  rivals, within-nest counts, Hausman fares, differentiation instruments, and (from notebook 02b) cost shifters.
* `linear_iv` estimates the plain logit and nested logit as linear IV-GMM (Berry 1994 inversion) with route
  fixed effects absorbed and quarter and carrier dummies.
* `make_problem` / `solve_rc` set up PyBLP random-coefficient problems: lognormal price coefficient
  (as in the course notes), optional income demographics or discrete business/leisure types, and an optional
  supply side.
* `evaluate_fixed` returns PyBLP results at given parameters on any sample (used for post-estimation and simulation).
* `post_estimation`, `simulate_merger`, `route_summary` compute elasticities, diversion, markups, and
  post-merger Bertrand–Nash equilibria.

In [ ]:
import pyblp
from linearmodels.iv import IVGMM
from linearmodels.iv.absorbing import AbsorbingLS

pyblp.options.verbose = False
pyblp.options.digits = 4

X1_VARS = ["prices", "nonstop", "distance", "hub_presence"]
EXOG = ["nonstop", "distance", "hub_presence"]
FE_ABSORB = "C(route)"
FE_DUMMIES = " + C(quarter) + C(carrier_fe)"
RC_DRAWS = 50

## Instruments

In [ ]:
def _loo_mean(df, value, keys, weight=None):
    """Leave-one-market-out mean of `value` within `keys` (all products of the own market excluded)."""
    w = df[weight] if weight else pd.Series(1.0, index=df.index)
    tot = (df[value] * w).groupby([df[k] for k in keys]).transform("sum")
    cnt = w.groupby([df[k] for k in keys]).transform("sum")
    own_tot = (df[value] * w).groupby([df[k] for k in keys + ["market_ids"]]).transform("sum")
    own_cnt = w.groupby([df[k] for k in keys + ["market_ids"]]).transform("sum")
    return (tot - own_tot) / (cnt - own_cnt)

BLP_IVS = ["iv_rival_nonstop", "iv_rival_distance", "iv_rival_presence", "iv_own_other"]
COUNT_IVS = ["iv_n_rivals", "iv_n_rival_products"]
NEST_IVS = ["iv_n_nest", "iv_rival_presence_nest"]
HAUSMAN_IVS = ["iv_hausman"]
DIFF_IVS = ["iv_diff_distance", "iv_diff_presence"]
COST_IVS = ["cs_fuel", "cs_pilot"]
IV_SETS = {
    "baseline": BLP_IVS + COUNT_IVS + NEST_IVS,
    "blp only": BLP_IVS + NEST_IVS,
    "differentiation": DIFF_IVS + COUNT_IVS + NEST_IVS,
    "baseline + Hausman": BLP_IVS + COUNT_IVS + NEST_IVS + HAUSMAN_IVS,
    "Hausman only": HAUSMAN_IVS + ["iv_n_nest"],
    "baseline + cost shifters": BLP_IVS + COUNT_IVS + NEST_IVS + COST_IVS,
    "cost shifters only": COST_IVS + ["iv_n_nest"],
}

def add_instruments(p):
    p = p.copy()
    g = p.groupby("market_ids")
    gf = p.groupby(["market_ids", "carrier"])
    gn = p.groupby(["market_ids", "nonstop"])
    gfn = p.groupby(["market_ids", "carrier", "nonstop"])
    for v, name in [("nonstop", "nonstop"), ("distance", "distance"), ("hub_presence", "presence")]:
        p[f"iv_rival_{name}"] = g[v].transform("sum") - gf[v].transform("sum")
    p["iv_own_other"] = gf["hub_presence"].transform("size") - 1          # carrier's other product in the market
    p["iv_n_rivals"] = p.n_carriers - 1
    p["iv_n_rival_products"] = g["prices"].transform("size") - gf["prices"].transform("size")
    p["iv_n_nest"] = gn["prices"].transform("size") - 1
    p["iv_rival_presence_nest"] = gn["hub_presence"].transform("sum") - gfn["hub_presence"].transform("sum")
    # Hausman: same carrier and service type, same quarter, other routes in the same distance band
    p["dist_band"] = pd.cut(p.route_distance, [0, 0.5, 1.0, 1.5, 2.0, 2.5, 10], labels=False)
    h = _loo_mean(p, "prices", ["carrier", "nonstop", "t", "dist_band"], weight="passengers")
    h2 = _loo_mean(p, "prices", ["carrier", "nonstop", "t"], weight="passengers")
    h3 = _loo_mean(p, "prices", ["nonstop", "t", "dist_band"], weight="passengers")
    p["iv_hausman"] = h.fillna(h2).fillna(h3)
    # Gandhi-Houde differentiation instruments: rival products within one s.d. in characteristic space
    m = p[["market_ids", "carrier", "distance", "hub_presence"]].reset_index()
    pairs = m.merge(m, on="market_ids", suffixes=("", "_r"))
    pairs = pairs[pairs.carrier != pairs.carrier_r]
    for v, name in [("distance", "distance"), ("hub_presence", "presence")]:
        close = (np.abs(pairs[v] - pairs[f"{v}_r"]) < p[v].std()).astype(float)
        p[f"iv_diff_{name}"] = close.groupby(pairs["index"]).sum().reindex(p.index).fillna(0).values
    return p

def set_demand_instruments(p, iv_cols, prefix="demand_instruments"):
    p = p.drop(columns=[c for c in p.columns if c.startswith(prefix)])
    for i, c in enumerate(iv_cols):
        p[f"{prefix}{i}"] = p[c].astype(float)
    return p

## Logit and nested logit by linear IV-GMM

$\ln s_j - \ln s_0 = x_j\beta + \alpha p_j + \rho \ln s_{j|g} + \xi_j$. Route fixed effects are removed by
within-route demeaning (exact for one dimension); quarter and carrier effects enter as dummies. Two-step
efficient GMM with route-clustered weights and standard errors.

In [ ]:
def demean_multi(df, groups, tol=1e-9, maxit=500):
    """Residualise columns of `df` on several sets of fixed effects by alternating projections."""
    X = df.astype(float).values.copy()
    codes = [pd.Series(np.asarray(g)).astype("category").cat.codes.values for g in groups]
    for _ in range(maxit):
        X0 = X.copy()
        for c in codes:
            n = np.bincount(c)
            for k in range(X.shape[1]):
                X[:, k] -= (np.bincount(c, weights=X[:, k]) / n)[c]
        if len(codes) == 1 or np.abs(X - X0).max() < tol:
            break
    return pd.DataFrame(X, columns=df.columns, index=df.index)

def nest_share(p, nest):
    if nest == "service":
        return np.log(p.shares / p.groupby(["market_ids", "nonstop"]).shares.transform("sum"))
    if nest == "air":
        return np.log(p.shares / p.groupby("market_ids").shares.transform("sum"))
    raise ValueError(nest)

def linear_iv(p, iv_cols, nest=None, fe=("route",), dummies=("quarter", "carrier_fe"), rho_fixed=None):
    """Logit (nest=None) or nested logit (nest='service' or 'air') by two-step IV-GMM."""
    s0 = 1 - p.groupby("market_ids").shares.transform("sum")
    d = pd.DataFrame({"y": np.log(p.shares) - np.log(s0)}, index=p.index)
    endog = ["prices"]
    if nest is not None:
        d["lws"] = nest_share(p, nest)
        if rho_fixed is None:
            endog.append("lws")
        else:
            d["y"] = d.y - rho_fixed * d.lws
    D = pd.get_dummies(p[list(dummies)].astype(str), drop_first=True).astype(float) if dummies else pd.DataFrame(index=p.index)
    allv = pd.concat([d, p[["prices"] + EXOG + list(iv_cols)], D], axis=1)
    allv = allv.loc[:, ~allv.columns.duplicated()]
    R = demean_multi(allv, [p[f] for f in fe])
    exog_cols = [c for c in EXOG + list(D.columns) if R[c].std() > 1e-8]
    cl = p.route.astype("category").cat.codes
    res = IVGMM(R.y, R[exog_cols], R[endog], R[list(iv_cols)], weight_type="clustered", clusters=cl) \
        .fit(cov_type="clustered", clusters=cl)
    out = {"alpha": float(res.params["prices"]), "alpha_se": float(res.std_errors["prices"]),
           "rho": float(res.params["lws"]) if "lws" in endog else rho_fixed,
           "rho_se": float(res.std_errors["lws"]) if "lws" in endog else None,
           "beta": {c: float(res.params[c]) for c in EXOG if c in res.params},
           "beta_se": {c: float(res.std_errors[c]) for c in EXOG if c in res.std_errors},
           "J": float(res.j_stat.stat), "J_df": int(len(iv_cols) - len(endog)), "N": int(res.nobs), "res": res}
    try:
        diag = res.first_stage.diagnostics
        out["first_stage"] = {k: {"F": float(diag.loc[k, "f.stat"]), "partial R2": float(diag.loc[k, "partial.rsquared"]),
                                  "Shea R2": float(diag.loc[k, "shea.rsquared"])} for k in diag.index}
    except Exception:
        out["first_stage"] = {}
    return out

## PyBLP problems

Random-coefficient utility: $u_{ijt} = \delta_{jt} - \alpha_i p_{jt} + \sigma_0 \nu_{i0} + \sigma_{ns}\nu_{i2}\,\text{nonstop}_{jt} + \varepsilon_{ijt}$
with a lognormal price coefficient $\alpha_i = \exp(\pi_0 + \pi_y y_i + \sigma_p \nu_{i1})$ (the course notes' BLP
specification). $\pi_0$ is the log of the median price sensitivity; $y_i$ is optional log income or a
business-traveller indicator. The constant's random coefficient plays the role of an "air travel" nest.

In [ ]:
def build_agents(p, kind="normal", draws=RC_DRAWS, seed=0, business_share=0.3, income=None):
    """Agent data for PyBLP. kind: 'normal' (nodes only), 'income' (nodes + log income), 'types' (2 agents)."""
    mk = np.unique(p.market_ids)
    T = len(mk)
    if kind == "types":
        a = pd.DataFrame({"market_ids": np.repeat(mk, 2), "weights": np.tile([business_share, 1 - business_share], T),
                          "const": 1.0, "business": np.tile([1.0, 0.0], T)})
        for k in range(3):
            a[f"nodes{k}"] = 0.0
        return a
    rng = np.random.default_rng(seed)
    a = pd.DataFrame({"market_ids": np.repeat(mk, draws), "weights": 1.0 / draws, "const": 1.0})
    nodes = rng.standard_normal((T * draws, 3))
    for k in range(3):
        a[f"nodes{k}"] = nodes[:, k]
    if kind == "income":
        brackets, edges = income
        first = p.groupby("market_ids")[["cbsa_a", "cbsa_b", "pop_a", "pop_b"]].first().loc[mk]
        Pa = brackets.reindex(first.cbsa_a).fillna(brackets.mean()).values
        Pb = brackets.reindex(first.cbsa_b).fillna(brackets.mean()).values
        w = (first.pop_a / (first.pop_a + first.pop_b)).values[:, None]
        cdf = np.cumsum(w * Pa + (1 - w) * Pb, axis=1)
        u = rng.random((T, draws))
        k = np.minimum((u[:, :, None] > cdf[:, None, :]).sum(axis=2), len(edges) - 2)
        lo, hi = np.array(edges[:-1])[k], np.array(edges[1:])[k]
        inc = lo + (hi - lo) * rng.random((T, draws))
        a["log_income"] = (np.log(np.maximum(inc, 2.5)) - np.log(75.0)).ravel()   # log of income relative to $75k
    return a

def make_problem(p, model, iv_cols, supply=False, agents=None, draws=RC_DRAWS, seed=0, business_share=0.3, income=None,
                 cost_vars=("cs_fuel", "cs_pilot", "cs_exfuel"), supply_ivs=None):
    """model in {'logit', 'nested_service', 'nested_air', 'rc', 'rc_income', 'rc_types'}."""
    pdata = set_demand_instruments(p, iv_cols).assign(firm_ids=p.carrier, clustering_ids=p.route)
    pdata = pdata.drop(columns=["nesting_ids"], errors="ignore")
    # carriers seen on fewer than 5 routes of this sample share the "OTH" fixed effect (avoids collinear dummies)
    nroutes = pdata.groupby("carrier_fe").route.nunique()
    pdata["carrier_fe"] = np.where(pdata.carrier_fe.map(nroutes) < 5, "OTH", pdata.carrier_fe)
    if model in ("logit", "nested_service", "nested_air"):
        x1 = pyblp.Formulation("0 + " + " + ".join(X1_VARS) + FE_DUMMIES, absorb=FE_ABSORB)
        if model == "nested_service":
            pdata["nesting_ids"] = pdata.nonstop
        if model == "nested_air":
            pdata["nesting_ids"] = 1
        return pyblp.Problem(x1, pdata)
    x1 = pyblp.Formulation("0 + " + " + ".join(EXOG) + FE_DUMMIES, absorb=FE_ABSORB)
    x2 = pyblp.Formulation("1 + I(-prices) + nonstop")
    forms = [x1, x2]
    if supply:
        x3 = pyblp.Formulation("0 + " + " + ".join(list(cost_vars) + EXOG) + FE_DUMMIES, absorb=FE_ABSORB)
        forms.append(x3)
        pdata = set_demand_instruments(pdata, supply_ivs or iv_cols, prefix="supply_instruments")
    kind = {"rc": "normal", "rc_income": "income", "rc_types": "types"}[model]
    if agents is None:
        agents = build_agents(p, kind, draws=draws, seed=seed, business_share=business_share, income=income)
    agent_form = {"normal": "0 + const", "income": "0 + const + log_income", "types": "0 + const + business"}[kind]
    return pyblp.Problem(tuple(forms), pdata, agent_formulation=pyblp.Formulation(agent_form), agent_data=agents,
                         rc_types=["linear", "log", "linear"], costs_type="log" if supply else "linear")

def rc_config(model, sigma0=(0.5, 0.3, 0.5), pi0=None):
    """Starting values and bounds for the nonlinear parameters of the RC models."""
    if model == "rc_types":   # discrete business / leisure types, no continuous heterogeneity
        sigma = np.zeros((3, 3)); sb = (np.zeros((3, 3)), np.zeros((3, 3)))
        pi = np.array([[0.0, 0.5], [np.log(1.5), -0.5], [0.0, 0.5]]) if pi0 is None else np.asarray(pi0)
        pb = (np.array([[0, -10], [-5, -10], [0, -10]]), np.array([[0, 10], [5, 10], [0, 10]]))
        return sigma, sb, pi, pb
    sigma = np.diag(sigma0); sb = (np.zeros((3, 3)), np.diag([8.0, 3.0, 8.0]))
    if model == "rc":
        pi = np.array([[0.0], [np.log(1.5)], [0.0]]) if pi0 is None else np.asarray(pi0)
        pb = (np.array([[0], [-5], [0]]), np.array([[0], [5], [0]]))
    else:  # rc_income: income shifts price sensitivity and the taste for nonstop service
        pi = np.array([[0.0, 0.0], [np.log(1.5), -0.3], [0.0, 0.0]]) if pi0 is None else np.asarray(pi0)
        pb = (np.array([[0, 0], [-5, -5], [0, -5]]), np.array([[0, 0], [5, 5], [0, 5]]))
    return sigma, sb, pi, pb

def solve_rc(problem, model, sigma0=(0.5, 0.3, 0.5), pi0=None, method="2s", supply=False, optimization=None,
             W=None, initial_update=None):
    sigma, sb, pi, pb = rc_config(model, sigma0, pi0)
    kw = dict(sigma=sigma, sigma_bounds=sb, pi=pi, pi_bounds=pb, method=method,
              optimization=optimization or pyblp.Optimization("l-bfgs-b", {"gtol": 1e-6, "maxiter": 200}),
              iteration=pyblp.Iteration("squarem", {"atol": 1e-12, "max_evaluations": 5000}),
              se_type="clustered", W_type="clustered")
    if supply:
        kw["costs_bounds"] = (1e-3, None)
    if W is not None:
        kw["W"] = W
    if initial_update is not None:
        kw["initial_update"] = initial_update
    t0 = time.time()
    res = problem.solve(**kw)
    res.elapsed = time.time() - t0
    return res

def objective(res):
    return float(np.asarray(res.objective).ravel()[0])

def rc_params(res):
    return {"sigma": np.diag(res.sigma).tolist(), "pi": np.asarray(res.pi).tolist(),
            "sigma_se": np.diag(res.sigma_se).tolist(), "pi_se": np.asarray(res.pi_se).tolist(),
            "beta": dict(zip(res.beta_labels, res.beta[:, 0].tolist())), "objective": objective(res),
            "converged": bool(res.converged)}

## Evaluating a model at fixed parameters on any sample

In [ ]:
def evaluate_fixed(p, model, params, iv_cols, draws=RC_DRAWS, seed=0, business_share=0.3, income=None, agents=None):
    """PyBLP results on data `p` with the parameters that govern substitution held fixed.

    logit / nested: params = {'alpha', 'rho'}; rc models: params = {'sigma', 'pi'}. Mean utilities are recovered by
    inversion on `p`; remaining linear parameters are concentrated out and do not affect prices or substitution."""
    # the concentrated-out linear parameters are irrelevant here, so collinearity in small samples is harmless
    saved = (pyblp.options.singular_tol, pyblp.options.collinear_atol, pyblp.options.collinear_rtol)
    pyblp.options.singular_tol, pyblp.options.collinear_atol, pyblp.options.collinear_rtol = np.inf, 0, 0
    try:
        return _evaluate_fixed(p, model, params, iv_cols, draws, seed, business_share, income, agents)
    finally:
        pyblp.options.singular_tol, pyblp.options.collinear_atol, pyblp.options.collinear_rtol = saved

def _evaluate_fixed(p, model, params, iv_cols, draws, seed, business_share, income, agents):
    problem = make_problem(p, model, iv_cols, draws=draws, seed=seed, business_share=business_share, income=income,
                           agents=agents)
    kw = dict(method="1s", optimization=pyblp.Optimization("return"), se_type="unadjusted",
              check_optimality="gradient")
    if model in ("logit", "nested_service", "nested_air"):
        K1 = problem.K1
        beta = np.full((K1, 1), np.nan); beta[0, 0] = params["alpha"]
        lb = np.full((K1, 1), -np.inf); ub = np.full((K1, 1), np.inf); lb[0, 0] = ub[0, 0] = params["alpha"]
        kw.update(beta=beta, beta_bounds=(lb, ub))
        if model.startswith("nested"):
            rho = float(np.clip(params["rho"], 0.0, 0.99))   # an estimate marginally below 0 is treated as 0 (logit)
            kw.update(rho=rho, rho_bounds=(rho, rho))
    else:
        S = np.diag(params["sigma"]) if np.ndim(params["sigma"]) == 1 else np.asarray(params["sigma"])
        P = np.asarray(params["pi"])
        kw.update(sigma=S, sigma_bounds=(S, S), pi=P, pi_bounds=(P, P),
                  iteration=pyblp.Iteration("squarem", {"atol": 1e-12, "max_evaluations": 5000}))
    return problem.solve(**kw)

def coef_rows(params_by_model):
    """Assemble a coefficient table: dict model -> list of (label, estimate, se)."""
    rows = {}
    for name, items in params_by_model.items():
        for label, b, se in items:
            rows.setdefault(label, {})[name] = f"{b:.3f}"
            rows.setdefault(label + " ", {})[name] = f"({se:.3f})" if se is not None and np.isfinite(se) else ""
    return pd.DataFrame(rows).T.fillna("")

## Post-estimation: elasticities, diversion ratios, costs, markups

In [ ]:
def post_estimation(res, p):
    out = p[["market_ids", "route", "quarter", "carrier", "nonstop", "prices", "shares", "passengers"]].copy()
    E = res.compute_elasticities()
    D = res.compute_diversion_ratios()
    costs = res.compute_costs()
    own_e, div0, div_best = np.empty(len(p)), np.empty(len(p)), np.empty(len(p))
    for m, idx in p.groupby("market_ids").indices.items():
        J = len(idx)
        e = E[idx, :J]; d = D[idx, :J]
        own_e[idx] = np.diag(e)
        div0[idx] = np.diag(d)                         # PyBLP stores diversion to the outside good on the diagonal
        dd = d.copy(); np.fill_diagonal(dd, -np.inf)
        div_best[idx] = dd.max(axis=1) if J > 1 else np.nan
    out["own_elasticity"] = own_e
    out["diversion_outside"] = div0
    out["diversion_top_rival"] = div_best
    out["mc"] = costs[:, 0]
    out["markup"] = out.prices - out.mc
    out["lerner"] = out.markup / out.prices
    return out

def summarize_post(pe):
    w = pe.passengers
    return {
        "own elasticity (pax-weighted mean)": float(np.average(pe.own_elasticity, weights=w)),
        "own elasticity (median)": float(pe.own_elasticity.median()),
        "Lerner index (median)": float(pe.lerner.median()),
        "Lerner index (pax-weighted mean)": float(np.average(pe.lerner.clip(-5, 5), weights=w)),
        "markup, $ (median)": float(pe.markup.median() * 100),
        "share with mc < 0": float((pe.mc < 0).mean()),
        "diversion to outside good (median)": float(pe.diversion_outside.median()),
        "diversion to closest rival (median)": float(np.nanmedian(pe.diversion_top_rival)),
    }

## Merger simulation

Marginal costs are recovered from the pre-merger Bertrand–Nash first-order conditions and held fixed (optionally
reduced by `efficiency` for the merging carriers). The merged firm prices all its products jointly; only markets
where both merging carriers are present can change.

In [ ]:
def simulate_merger(res, p, parties, markets=None, efficiency=0.0, costs=None, kappa=1.0):
    """kappa < 1 gives partial internalisation between the merging carriers (conduct experiments)."""
    acquirer = parties[0]
    if costs is None:
        costs = res.compute_costs()[:, 0]
    cost_post = np.where(p.carrier.isin(parties), costs * (1 - efficiency), costs)
    pos = p.groupby("market_ids").indices
    if markets is None:
        both = p.groupby("market_ids").carrier.agg(lambda s: set(parties) <= set(s))
        markets = both.index[both].tolist()
    new_prices = p.prices.values.copy()
    cs_pre, cs_post = {}, {}
    it = pyblp.Iteration("simple", {"atol": 1e-10, "max_evaluations": 20000})
    for m in markets:
        idx = pos[m]
        car = p.carrier.values[idx]
        try:
            if kappa == 1.0:
                firms = np.where(np.isin(car, parties), acquirer, car)
                pr = res.compute_prices(firm_ids=firms.reshape(-1, 1), costs=cost_post[idx].reshape(-1, 1), market_id=m, iteration=it)
            else:
                O = (car[:, None] == car[None, :]).astype(float)
                both = np.isin(car, parties)
                O[np.ix_(both, both)] = np.where(car[both][:, None] == car[both][None, :], 1.0, kappa)
                pr = res.compute_prices(ownership=O, costs=cost_post[idx].reshape(-1, 1), market_id=m, iteration=it)
            new_prices[idx] = pr[:, 0]
            cs_pre[m] = float(res.compute_consumer_surpluses(market_id=m)[0, 0])
            cs_post[m] = float(res.compute_consumer_surpluses(prices=pr, market_id=m)[0, 0])
        except Exception as e:
            print("failed", m, repr(e)[:200])
    sim = p[["market_ids", "route", "quarter", "carrier", "nonstop", "prices", "shares", "passengers", "market_size"]].copy()
    sim["prices_post"] = new_prices
    sim["dprice_pct"] = 100 * (sim.prices_post / sim.prices - 1)
    sim["mc"] = costs
    sim = sim[sim.market_ids.isin(markets)].copy()
    cs = pd.DataFrame({"cs_pre": pd.Series(cs_pre), "cs_post": pd.Series(cs_post)})
    cs.index.name = "market_ids"
    return sim, cs.reset_index()

def route_summary(sim, cs, parties):
    """Market-level passenger-weighted price changes and consumer-surplus change in dollars."""
    sim = sim.copy()
    sim["party"] = sim.carrier.isin(parties)
    def wavg(d, col):
        return np.average(d[col], weights=d.passengers) if len(d) else np.nan
    rows = []
    for m, d in sim.groupby("market_ids"):
        dp = d[d.party]
        rows.append({"market_ids": m, "route": d.route.iat[0], "quarter": d.quarter.iat[0],
                     "dprice_parties": wavg(dp, "dprice_pct"), "dprice_rivals": wavg(d[~d.party], "dprice_pct"),
                     "dprice_all": wavg(d, "dprice_pct"), "fare_parties": 100 * wavg(dp, "prices"),
                     "passengers": d.passengers.sum(), "party_share": dp.passengers.sum() / d.passengers.sum(),
                     "parties_nonstop": int(dp.nonstop.max()), "market_size": d.market_size.iat[0]})
    r = pd.DataFrame(rows).merge(cs, on="market_ids", how="left")
    # PyBLP consumer surplus is in price units ($100) per potential traveller
    r["dcs_usd"] = (r.cs_post - r.cs_pre) * 100 * r.market_size
    return r

def party_avg(rs, col="dprice_parties"):
    """Passenger-weighted mean over markets, weighting by the merging carriers' passengers."""
    w = rs.passengers * rs.party_share
    return float(np.average(rs[col], weights=w)) if len(rs) else np.nan